# Data Science Lab # 10
## Feature Engineering and Feature Selection

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load Dataset
dataset = pd.read_excel('Feature_Engineering_Dataset_100Rows.xlsx')
dataset.head()

,Student_ID,Study_Hours,Family_Income,Gender,Department,Attendance,Previous_GPA,Final_Result
0,1,7,142938,Male,CS,54,2.90,Pass
1,2,5,102513,Male,IT,100,3.68,Pass
2,3,7,93988,Female,CS,81,3.98,Pass
3,4,10,77273,Female,IT,52,3.86,Pass
4,5,9,148852,Female,IT,55,3.47,Pass


## Part A: Feature Engineering
### Task 1: Logarithmic Transformation

In [2]:
# Apply logarithmic transformation on Family_Income
dataset['Log_Income'] = np.log1p(dataset['Family_Income'])

# Display original and transformed values
dataset[['Family_Income', 'Log_Income']].head()

,Family_Income,Log_Income
0,142938,11.870173
1,102513,11.537755
2,93988,11.450933
3,77273,11.255113
4,148852,11.910715


**Questions**
1. **Why is logarithmic transformation applied?**
   Logarithmic transformation is often used to reduce skewness in data, normalize the distribution of features with a wide range of values, and make the relationships between variables more linear.
2. **What effect does it have on large values?**
   It compresses the scale of large values, reducing the impact of outliers and making large differences smaller, thereby bringing all values closer together.

### Task 2: Label Encoding

In [3]:
from sklearn.preprocessing import LabelEncoder

le_gender = LabelEncoder()
le_result = LabelEncoder()

dataset['Gender_Encoded'] = le_gender.fit_transform(dataset['Gender'])
dataset['Result_Encoded'] = le_result.fit_transform(dataset['Final_Result'])

# Display encoded values
dataset[['Gender', 'Gender_Encoded', 'Final_Result', 'Result_Encoded']].head()

,Gender,Gender_Encoded,Final_Result,Result_Encoded
0,Male,1,Pass,1
1,Male,1,Pass,1
2,Female,0,Pass,1
3,Female,0,Pass,1
4,Female,0,Pass,1


**Questions**
1. **Which numeric values were assigned to Male and Female?**
   Female is assigned 0, and Male is assigned 1 (alphabetical order).
2. **Which numeric values were assigned to Pass and Fail?**
   Fail is assigned 0, and Pass is assigned 1.

### Task 3: One-Hot Encoding

In [4]:
# Apply One-Hot Encoding on Department
dataset = pd.get_dummies(dataset, columns=['Department'], prefix='Department', dtype=int)

# Display the transformed dataset
dataset.head()

,Student_ID,Study_Hours,Family_Income,Gender,Attendance,Previous_GPA,Final_Result,Log_Income,Gender_Encoded,Result_Encoded,Department_CS,Department_IT,Department_SE
0,1,7,142938,Male,54,2.90,Pass,11.870173,1,1,1,0,0
1,2,5,102513,Male,100,3.68,Pass,11.537755,1,1,0,1,0
2,3,7,93988,Female,81,3.98,Pass,11.450933,0,1,1,0,0
3,4,10,77273,Female,52,3.86,Pass,11.255113,0,1,0,1,0
4,5,9,148852,Female,55,3.47,Pass,11.910715,0,1,0,1,0


**Questions**
1. **Why is One-Hot Encoding used for nominal categories?**
   It prevents the machine learning algorithm from assuming a false ordinal relationship (e.g., 0 < 1 < 2) between nominal categories that have no inherent rank.
2. **What problem can occur if Label Encoding is used instead?**
   The model might interpret the categorical labels as having an order or mathematical magnitude, which can lead to biased or incorrect predictions.

### Task 4: Feature Interaction

In [5]:
# Create Study_Attendance feature
dataset['Study_Attendance'] = dataset['Study_Hours'] * dataset['Attendance']

# Display the first 10 records
dataset[['Study_Hours', 'Attendance', 'Study_Attendance']].head(10)

,Study_Hours,Attendance,Study_Attendance
0,7,54,378
1,5,100,500
2,7,81,567
3,10,52,520
4,9,55,495
5,2,71,142
6,10,82,820
7,4,47,188
8,5,46,230
9,9,46,414


**Questions**
1. **What is Feature Interaction?**
   Feature interaction is the creation of a new feature by combining two or more existing features (usually via multiplication or addition) to represent their combined effect on the target variable.
2. **How can interaction features improve prediction accuracy?**
   They can capture non-linear relationships and synergies between variables that individual features cannot explain alone (e.g., studying more hours might only be highly effective if attendance is also high).

### Task 5: Polynomial Features

In [6]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(dataset[['Study_Hours', 'Previous_GPA']])

poly_df = pd.DataFrame(poly_features, columns=poly.get_feature_names_out(['Study_Hours', 'Previous_GPA']))

# Display generated features
poly_df.head()

,Study_Hours,Previous_GPA,Study_Hours^2,Study_Hours Previous_GPA,Previous_GPA^2
0,7.0,2.90,49.0,20.30,8.4100
1,5.0,3.68,25.0,18.40,13.5424
2,7.0,3.98,49.0,27.86,15.8404
3,10.0,3.86,100.0,38.60,14.8996
4,9.0,3.47,81.0,31.23,12.0409


**Questions**
1. **Why are Polynomial Features useful?**
   They allow linear models to capture non-linear patterns by explicitly adding non-linear combinations and powers of features.
2. **What type of relationships can they capture?**
   Quadratic, cubic, and other non-linear interactive relationships between the independent features.

## Part B: Feature Selection
### Data Preparation

In [7]:
# Separate features (X) and target (y)
# We drop original columns that we encoded or don't need for selection
features_to_drop = ['Student_ID', 'Gender', 'Final_Result', 'Family_Income', 'Result_Encoded']
X = dataset.drop(columns=[col for col in features_to_drop if col in dataset.columns])
y = dataset['Result_Encoded']

X.head()

,Study_Hours,Attendance,Previous_GPA,Log_Income,Gender_Encoded,Department_CS,Department_IT,Department_SE,Study_Attendance
0,7,54,2.90,11.870173,1,1,0,0,378
1,5,100,3.68,11.537755,1,0,1,0,500
2,7,81,3.98,11.450933,0,1,0,0,567
3,10,52,3.86,11.255113,0,0,1,0,520
4,9,55,3.47,11.910715,0,0,1,0,495


### Task 6: Chi-Square Test (Filter Method)

In [8]:
from sklearn.feature_selection import chi2, SelectKBest

# Apply Chi-Square Feature Selection
chi2_selector = SelectKBest(score_func=chi2, k=3)
X_kbest = chi2_selector.fit_transform(X, y)

chi_scores = pd.DataFrame({'Feature': X.columns, 'Chi2_Score': chi2_selector.scores_})
chi_scores = chi_scores.sort_values(by='Chi2_Score', ascending=False)

print("Chi-Square Scores and Ranked Features:")
display(chi_scores)

print("\nTop 3 Selected Features:")
top_3_features = chi_scores.head(3)['Feature'].tolist()
print(top_3_features)

Chi-Square Scores and Ranked Features:


,Feature,Chi2_Score
8,Study_Attendance,8343.958947
0,Study_Hours,87.254238
1,Attendance,42.476405
2,Previous_GPA,7.193081
6,Department_IT,2.697214
7,Department_SE,1.617130
4,Gender_Encoded,0.062325
5,Department_CS,0.055202
3,Log_Income,0.008455



Top 3 Selected Features:
['Study_Attendance', 'Study_Hours', 'Attendance']


**Questions**
1. **Which feature received the highest score?**
   *(Check the output above for the top feature)*
2. **Why is Chi-Square considered a Filter Method?**
   Because it evaluates the importance of each feature independently of any machine learning model, relying solely on statistical tests to filter out less relevant features.

### Task 7: Wrapper Method (RFE)

In [9]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)
rfe = RFE(estimator=log_reg, n_features_to_select=3)
rfe.fit(X, y)

rfe_rankings = pd.DataFrame({'Feature': X.columns, 'Ranking': rfe.ranking_})
rfe_rankings = rfe_rankings.sort_values(by='Ranking')

print("Feature Rankings:")
display(rfe_rankings)

print("\nSelected Features:")
rfe_selected = rfe_rankings[rfe_rankings['Ranking'] == 1]['Feature'].tolist()
print(rfe_selected)

Feature Rankings:


,Feature,Ranking
0,Study_Hours,1
2,Previous_GPA,1
4,Gender_Encoded,1
6,Department_IT,2
3,Log_Income,3
7,Department_SE,4
1,Attendance,5
5,Department_CS,6
8,Study_Attendance,7



Selected Features:
['Study_Hours', 'Previous_GPA', 'Gender_Encoded']


**Questions**
1. **Which features were selected?**
   *(Check output above for features with Rank 1)*
2. **Why is RFE called a Wrapper Method?**
   Because it wraps around a specific machine learning model (Logistic Regression here) and iteratively trains the model, evaluating feature subsets to find the best performing combination.

### Task 8: Embedded Method (LASSO)

In [10]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)
lasso.fit(X, y)

lasso_coeffs = pd.DataFrame({'Feature': X.columns, 'Coefficient': lasso.coef_})
features_retained = lasso_coeffs[lasso_coeffs['Coefficient'] != 0]['Feature'].tolist()
features_removed = lasso_coeffs[lasso_coeffs['Coefficient'] == 0]['Feature'].tolist()

print("Feature Coefficients:")
display(lasso_coeffs)

print("\nFeatures Retained by LASSO:")
print(features_retained)

print("\nFeatures Removed by LASSO:")
print(features_removed)

Feature Coefficients:


,Feature,Coefficient
0,Study_Hours,0.004193
1,Attendance,0.003142
2,Previous_GPA,0.000000
3,Log_Income,-0.000000
4,Gender_Encoded,-0.000000
5,Department_CS,0.000000
6,Department_IT,-0.000000
7,Department_SE,0.000000
8,Study_Attendance,0.001358



Features Retained by LASSO:
['Study_Hours', 'Attendance', 'Study_Attendance']

Features Removed by LASSO:
['Previous_GPA', 'Log_Income', 'Gender_Encoded', 'Department_CS', 'Department_IT', 'Department_SE']


**Questions**
1. **Which features were removed?**
   *(Check output above for features with coefficient exactly equal to 0)*
2. **How does LASSO automatically perform feature selection?**
   LASSO applies an L1 regularization penalty to the objective function, which penalizes the absolute magnitude of coefficients. This effectively shrinks the coefficients of less important features to exactly zero, thereby removing them from the model.